# PyTorch FoundationStereo · 800×960 · 40 组批处理

此 notebook 按 `fs_tensorrt800x960_batch.ipynb` 的流程处理 `data/` 中完整的 40 组双目数据，但使用本地 FoundationStereo checkpoint 的 PyTorch inference，而不是 TensorRT engine。

每组流程如下：

1. 校正原始 2448×2048 双目图像和 mask；
2. 将左右图缩放到固定 **800×960** PyTorch 输入网格；
3. 使用 PyTorch 直接运行一次 `model.forward(..., iters=32, test_mode=True)`；
4. 保存 800×960 原始视差为单通道 float32 TIFF（单位为该网格的像素），并保留投影、过滤及 mesh 面积统计，不导出 mesh.ply。

输出与 TensorRT 结果隔离，不会覆盖 `result/tensorrt_800x960/`：

- `result/pytorch_800x960/<group>/<capture>/comparision.jpg`：PyTorch 输入、mask、右图和 disparity；
- `result/pytorch_800x960/<group>/<capture>/disparity.tiff`：800×960 单通道 float32 原始视差，不归一化、不量化；
- `result/pytorch_800x960/areas.json`：各组 mesh 面积统计；


## 1. 环境、40 组输入与独立输出目录


In [1]:
%load_ext autoreload
%autoreload 2

from fs_high_resolution_utils import (
    bootstrap_notebook,
    build_constrained_mesh,
    load_rectify_and_select_mask,
    postprocess_disparity_gpu,
    run_foundation_stereo_inference,
)

REPO_ROOT, FS_ROOT, DEVICE = bootstrap_notebook(globals())
DATA_ROOT = REPO_ROOT / 'data'
OUTPUT_ROOT = REPO_ROOT / 'result' / 'pytorch_800x960'
MODEL_HEIGHT, MODEL_WIDTH = 800, 960
ORIGINAL_WIDTH, ORIGINAL_HEIGHT = 2448, 2048
REQUIRED_INPUTS = ('left.png', 'right.png', 'calibration.json', 'masks/left_mask.png')

def capture_sort_key(path):
    try:
        return (0, int(path.name))
    except ValueError:
        return (1, path.name)

CAPTURES = sorted(
    (capture_dir
     for group_dir in sorted(DATA_ROOT.iterdir()) if group_dir.is_dir()
     for capture_dir in sorted(group_dir.iterdir(), key=capture_sort_key)
     if capture_dir.is_dir() and all((capture_dir / item).is_file() for item in REQUIRED_INPUTS)),
    key=lambda path: (path.parent.name, capture_sort_key(path)),
)
if len(CAPTURES) != 40:
    raise RuntimeError(f'Expected 40 complete captures in {DATA_ROOT}, found {len(CAPTURES)}.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for capture_dir in CAPTURES:
    (OUTPUT_ROOT / capture_dir.relative_to(DATA_ROOT)).mkdir(parents=True, exist_ok=True)

print('Repository:', REPO_ROOT)
print('GPU:', torch.cuda.get_device_name(DEVICE))
print(f'Found {len(CAPTURES)} captures; PyTorch outputs: {OUTPUT_ROOT}')


def scaled_model_intrinsics(rectified_k, source_width, source_height):
    """Scale rectified intrinsics to the fixed disparity grid."""
    scale_x, scale_y = source_width / MODEL_WIDTH, source_height / MODEL_HEIGHT
    k_model = rectified_k.copy().astype(np.float32)
    k_model[0, :] /= scale_x
    k_model[1, :] /= scale_y
    k_model[2, :] = (0.0, 0.0, 1.0)
    return k_model


def save_pytorch_comparison(left_rgb, right_rgb, mask, disparity, output_path, capture_label):
    mask_overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    mask_overlay[mask] = (0.0, 1.0, 0.0, 0.45)
    finite = disparity[np.isfinite(disparity) & (disparity > 0)]
    vmax = float(np.percentile(finite, 99)) if finite.size else 1.0

    fig, axes = plt.subplots(2, 2, figsize=(16, 14), constrained_layout=True)
    try:
        axes[0, 0].imshow(left_rgb)
        axes[0, 0].set_title('Left input')
        axes[0, 1].imshow(left_rgb)
        axes[0, 1].imshow(mask_overlay)
        axes[0, 1].set_title('Left input + mask')
        axes[1, 0].imshow(right_rgb)
        axes[1, 0].set_title('Right input')
        disparity_image = axes[1, 1].imshow(
            disparity, cmap='turbo', vmin=0.0, vmax=vmax,
        )
        axes[1, 1].set_title('Disparity')
        fig.colorbar(disparity_image, ax=axes[1, 1], label='Disparity (pixels)')
        for axis in axes.flat:
            axis.axis('off')
        fig.suptitle(f'PyTorch FoundationStereo | {capture_label}')
        fig.savefig(output_path, dpi=150, format='jpg', pil_kwargs={'quality': 95})
    finally:
        plt.close(fig)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Repository: /workspace/FS_Engine
GPU: NVIDIA RTX PRO 4000 Blackwell
Found 40 captures; PyTorch outputs: /workspace/FS_Engine/result/pytorch_800x960


## 2. 只加载一次 PyTorch FoundationStereo 模型


In [2]:
# This is the same fixed-resolution, non-hierarchical model forward used for
# the 800×960 engine comparison.  The local checkpoint config selects ViT-L.
VALID_ITERS, HIERA = 32, 0

model, CKPT_PATH = load_foundation_stereo_model(FS_ROOT, DEVICE)
print(f'Model loaded: {CKPT_PATH}')
print(f'Input grid: {MODEL_WIDTH}×{MODEL_HEIGHT} | valid_iters={VALID_ITERS} | hiera={HIERA}')
print({key: f'{value:.1f} MiB' for key, value in gpu_memory_mib().items()})


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:29: UserWarning: xFormers is disabled (Attention)
  warnings.warn("xFormers is disabled (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:35: UserWarning: xFormers is disabled (Block)
  warnings.warn("xFormers is disabled (Block)")
/root/.cache/torch/hub/f

Model loaded: /workspace/FS_Engine/weight/23-51-11/model_best_bp2.pth
Input grid: 960×800 | valid_iters=32 | hiera=0
{'allocated': '1431.9 MiB', 'reserved': '1436.0 MiB', 'peak_allocated': '1431.9 MiB', 'peak_reserved': '1436.0 MiB'}


## 3. 批量 PyTorch inference、真实比例投影、导出 comparison / float32 TIFF / areas.json


In [3]:
Z_FAR_M = 1.0
REMOVE_INVISIBLE, DENOISE_CLOUD = True, True
DENOISE_MAX_NEIGHBOR_DISTANCE_M = 0.01
DENOISE_MIN_NEIGHBORS, DENOISE_EDGE_MIN_NEIGHBORS = 3, 2
DOWNSAMPLE_PIXELS = 1  # Build mesh from every retained point on the 800×960 grid.
MESH_MAX_EDGE_M, MESH_MAX_DEPTH_JUMP_M = 0.02, 0.01

areas_m2 = {}
failures = []
total_inference_seconds = 0.0

for number, capture_dir in enumerate(CAPTURES, start=1):
    relative_path = capture_dir.relative_to(DATA_ROOT)
    group_name, capture_name = relative_path.parts
    output_dir = OUTPUT_ROOT / relative_path
    left_path = capture_dir / 'left.png'
    right_path = capture_dir / 'right.png'
    mask_path = capture_dir / 'masks' / 'left_mask.png'
    calibration_path = capture_dir / 'calibration.json'

    try:
        rectified_left, rectified_right, rectified_k, baseline_m, rectified_left_mask = (
            load_rectify_and_select_mask(
                globals(), left_path, right_path, mask_path, calibration_path,
                ORIGINAL_WIDTH, ORIGINAL_HEIGHT,
            )
        )
        source_height, source_width = rectified_left.shape[:2]
        if (source_width, source_height) != (ORIGINAL_WIDTH, ORIGINAL_HEIGHT):
            raise RuntimeError(f'Unexpected rectified size: {source_width}×{source_height}')

        model_left = cv2.resize(rectified_left, (MODEL_WIDTH, MODEL_HEIGHT), interpolation=cv2.INTER_LINEAR)
        model_right = cv2.resize(rectified_right, (MODEL_WIDTH, MODEL_HEIGHT), interpolation=cv2.INTER_LINEAR)
        model_mask = cv2.resize(
            rectified_left_mask.astype(np.uint8), (MODEL_WIDTH, MODEL_HEIGHT), interpolation=cv2.INTER_NEAREST
        ).astype(bool)
        k_model = scaled_model_intrinsics(rectified_k, source_width, source_height)

        # The first capture performs one discarded warmup before its timed inference.
        inference = run_foundation_stereo_inference(
            model, model_left, model_right, DEVICE, VALID_ITERS, HIERA,
            warmup_runs=1 if number == 1 else 0,
        )
        total_inference_seconds += inference.seconds
        disparity = np.ascontiguousarray(
            inference.disparity.detach().cpu().numpy(), dtype=np.float32
        )
        if disparity.shape != (MODEL_HEIGHT, MODEL_WIDTH) or not np.isfinite(disparity).all():
            raise RuntimeError(f'Invalid PyTorch disparity: shape={disparity.shape}')

        # Preserve raw disparity values as a single-channel float32 TIFF.
        disparity_path = output_dir / 'disparity.tiff'
        if not cv2.imwrite(str(disparity_path), disparity):
            raise OSError(f'Failed to write disparity TIFF: {disparity_path}')

        surface = postprocess_disparity_gpu(
            inference.disparity, k_model, baseline_m, model_mask, DEVICE, Z_FAR_M,
            remove_invisible=REMOVE_INVISIBLE,
            denoise=DENOISE_CLOUD,
            max_neighbor_distance_m=DENOISE_MAX_NEIGHBOR_DISTANCE_M,
            min_neighbors=DENOISE_MIN_NEIGHBORS,
            edge_min_neighbors=DENOISE_EDGE_MIN_NEIGHBORS,
        )
        mesh_result = build_constrained_mesh(
            surface.xyz_map, surface.keep_mask, model_mask, model_left,
            downsample_pixels=DOWNSAMPLE_PIXELS,
            max_edge_m=MESH_MAX_EDGE_M,
            max_depth_jump_m=MESH_MAX_DEPTH_JUMP_M,
        )
        save_pytorch_comparison(
            model_left, model_right, model_mask, disparity,
            output_dir / 'comparision.jpg', str(relative_path),
        )
        areas_m2.setdefault(group_name, {})[capture_name] = mesh_result.area_m2
        print(
            f'[{number:02d}/{len(CAPTURES)}] {relative_path}: '
            f'{mesh_result.area_m2:.6f} m² ({mesh_result.area_m2 * 1e4:.2f} cm²), '
            f'{inference.seconds * 1000:.1f} ms'
        )
    except Exception as error:
        failures.append((str(relative_path), f'{type(error).__name__}: {error}'))
        print(f'[{number:02d}/{len(CAPTURES)}] FAILED {relative_path}: {error}')
    finally:
        for variable_name in (
            'rectified_left', 'rectified_right', 'rectified_k', 'baseline_m', 'rectified_left_mask',
            'model_left', 'model_right', 'model_mask', 'k_model', 'inference', 'disparity',
            'surface', 'mesh_result',
        ):
            globals().pop(variable_name, None)
        torch.cuda.empty_cache()

if failures:
    formatted = '\n'.join(f' - {path}: {message}' for path, message in failures)
    raise RuntimeError(f'{len(failures)} capture(s) failed; areas.json was not written:\n{formatted}')

areas_path = OUTPUT_ROOT / 'areas.json'
areas_payload = {
    'unit': 'm²',
    'runtime': 'PyTorch',
    'checkpoint': str(CKPT_PATH.relative_to(REPO_ROOT)),
    'input_grid': [MODEL_HEIGHT, MODEL_WIDTH],
    'valid_iters': VALID_ITERS,
    'hiera': HIERA,
    'mesh_downsample_pixels': DOWNSAMPLE_PIXELS,
    'areas_m2': areas_m2,
}
areas_path.write_text(json.dumps(areas_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Completed {len(CAPTURES)} captures in {total_inference_seconds:.1f} s of PyTorch inference.')
print(f'Areas JSON: {areas_path}')


/workspace/FS_Engine/FoundationStereo/core/submodule.py:390: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


[01/40] Volunteer2_lower/0: 0.018224 m² (182.24 cm²), 995.5 ms


/workspace/FS_Engine/FoundationStereo/core/submodule.py:390: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


[02/40] Volunteer2_lower/1: 0.018863 m² (188.63 cm²), 990.9 ms


/workspace/FS_Engine/FoundationStereo/core/submodule.py:390: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


...

[38/40] Volunteer3_upper/7: 0.084024 m² (840.24 cm²), 993.9 ms


/workspace/FS_Engine/FoundationStereo/core/submodule.py:390: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


[39/40] Volunteer3_upper/8: 0.083565 m² (835.65 cm²), 996.3 ms


/workspace/FS_Engine/FoundationStereo/core/submodule.py:390: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


[40/40] Volunteer3_upper/9: 0.087678 m² (876.78 cm²), 993.0 ms
Completed 40 captures in 39.9 s of PyTorch inference.
Areas JSON: /workspace/FS_Engine/result/pytorch_800x960/areas.json


## 输出位置

成功完成后，40 组结果都在 `result/pytorch_800x960/` 下，每组的 `comparision.jpg` 展示左图、mask 叠加图、右图和视差图，`disparity.tiff` 为 800×960 单通道 float32 原始视差，不再写出 `mesh.ply`；`areas.json` 的目录结构与 `data/<group>/<capture>/` 一致。可与 `result/tensorrt_800x960/` 的同一组目录逐一对比。
